# Exercise: keeping .Stat Suite in sync with FMR

Publish real ECB exchange-rate structures to an FMR and from the FMR to
.Stat Suite, load data into .Stat, then remove from .Stat the structures
that the FMR no longer has. The exercise is a closed loop on .Stat: it
does not touch what was already there, and at the end it removes the
data and the structures that it has published. The course notebook, `fmr_stat_sync.ipynb`, explains the
steps.

The exercise takes about 20 minutes. Run the cells in order. Six steps
are marked *to complete*: their cell is empty, and the notes and the
hint say what to write. The other cells are already written. The answer
is in `fmr_stat_sync_practice.py`, in the folder of this notebook.

## Setup

The configuration and the helpers are in `helpers.py`, in the folder of
this notebook. The settings are read from environment variables when it
is imported, so they must be set before the imports. Each participant
has an SDMX Lab instance, with its user and its password: the second
cell sets `PREFIX` and requests them, unless they are already set in the
environment; the password is not displayed. The kernel must be restarted
to change a setting after the imports.

| Variable | Meaning | Default |
|---|---|---|
| `SDMX_LAB` | Identifier of the SDMX Lab instance, the first part of its address (`<id>.sdmx.solutions`) | requested |
| `SDMX_LAB_USER` and `SDMX_LAB_PASSWORD` | User and password of the instance | requested |
| `PREFIX` | Prefix of the ids of the structures | `PX_SYNC` |
| `AGENCY` | Agency of the structures | `BIS` |
| `SPACE` | Dataspace of .Stat | `demo-design` |

The addresses of the FMR, of the .Stat services and of Keycloak are
derived from the instance.

`fmr_client()` and `stat_uploader()` return the clients with write
access, `fmr_service()` and `stat_service()` the readers, and
`build_structures(ecb)` the structures to publish.

In [ ]:
%pip install --quiet ./pysdmx-1.24.0rc1-py3-none-any.whl[all]
%pip install --quiet certifi

In [ ]:
import os
from getpass import getpass

# Every structure of the tutorial starts with this prefix, which tells them
# apart from the other structures of the agency. Only letters, digits and
# underscores are allowed.
os.environ["PREFIX"] = "PX_SYNC"

# The SDMX Lab instance of the participant, its user and its password are
# requested, unless they are already set in the environment. The password
# is not displayed.
if not os.environ.get("SDMX_LAB"):
    os.environ["SDMX_LAB"] = input("SDMX Lab instance (<id>.sdmx.solutions): ")
if not os.environ.get("SDMX_LAB_USER"):
    os.environ["SDMX_LAB_USER"] = input("User: ")
    os.environ["SDMX_LAB_PASSWORD"] = getpass("Password: ")

In [ ]:
# Some imports stay unused, and some names undefined, until the exercise
# is completed.
# ruff: noqa: F401, F821
from contextlib import suppress
from io import BytesIO

from helpers import (
    AGENCY,
    DELETION_ORDER,
    DLM,
    ECB_STRUCTURES,
    PREFIX,
    STAT_FORMAT,
    build_dataset,
    build_structures,
    check_connections,
    fmr_client,
    fmr_service,
    list_urns,
    stat_data,
    stat_service,
    stat_uploader,
)

from pysdmx.api.fmr.maintenance import StructureAction
from pysdmx.api.qb import (
    StructureDetail,
    StructureQuery,
    StructureReference,
    StructureType,
)
from pysdmx.errors import Invalid, NotFound
from pysdmx.io import read_sdmx
from pysdmx.model import AgencyScheme, Code, Codelist

## Part 1 · Publish to the FMR

### Step 1 · Check the connections.

`check_connections` tests the FMR and .Stat with read-only queries. Both services must be connected before going on.

If a service fails, restart the kernel and run the cells again, to enter the instance, the user and the password again.

In [ ]:
check_connections()

### Step 2 · Read the ECB exchange-rate structures and prepare the structures to publish.

`build_structures` copies the ECB codelists and concepts to the agency of the participant, adds `PREFIX` to their ids, and builds two dataflows on top of them: EXR (in `keep`) and EXR_OLD (in `legacy`).

`legacy` stands for structures that have been retired in the FMR but remain on .Stat.

In [ ]:
ecb = read_sdmx(ECB_STRUCTURES, validate=False)
keep, legacy = build_structures(ecb)
print("keep:  ", [s.short_urn for s in keep])
print("legacy:", [s.short_urn for s in legacy])

### Step 3 (to complete) · Upload the structures to keep to the FMR.

`Replace` overwrites the existing structures, so the step can be run again. `legacy` is not uploaded.

**Hint:** `fmr_client()` returns a `RegistryMaintenanceClient`. Call its `put_structures` method with `keep` and `action=StructureAction.Replace`.

In [ ]:
# To be completed

### Step 4 (to complete) · Download the EXR dataflow with all its references from the FMR.

The dataflow comes with its DSD, the concept scheme, the codelists and the SDMX:AGENCIES agency scheme.

**Hint:** Build a `StructureQuery` with `artefact_type=StructureType.DATAFLOW`, `agency_id=AGENCY`, `resource_id=f"{PREFIX}_DF_EXR"`, `version="1.0"`, `detail=StructureDetail.FULL` and `references=StructureReference.ALL`. Send it with `fmr_service().structure(query)`, which returns bytes, and read them with `read_sdmx(BytesIO(...), validate=False)`.

**Result:** `graph`.

In [ ]:
# To be completed

## Part 2 · Publish from the FMR to .Stat and load data

### Step 5 · Record what .Stat already has, and find what it does not have yet.

The SDMX:AGENCIES agency scheme is left out: the agency scheme is not maintained by this process.

`before` records the structures of the tutorial that .Stat already has: they are not published again, and the exercise does not change or delete them. On a first run it is empty.

In [ ]:
before = list_urns(stat_service())
from_fmr = [s for s in graph.structures if not isinstance(s, AgencyScheme)]
to_stat = [s for s in from_fmr if s.short_urn not in before]
print([s.short_urn for s in to_stat])

### Step 6 (to complete) · Publish them to .Stat.

The result has `success`, which must be True, and `messages`: if .Stat rejects a structure, the reason is in the messages.

**Hint:** If `to_stat` is not empty, call `stat_uploader().submit_structure` with `to_stat` and `STAT_FORMAT`, and print the `success` and the `messages` of the result.

In [ ]:
# To be completed

### Step 7 · Build the dataset of the dataflow.

`build_dataset(graph)` reads monthly ECB reference exchange rates of four currencies and returns them as a dataset of the dataflow.

The data are loaded only into a dataflow that this exercise has published: otherwise `dataset` is None.

In [ ]:
flow = graph.get_dataflows()[0]
dataset = build_dataset(graph) if flow.short_urn not in before else None
print(len(dataset.data) if dataset is not None else 0, "observations to load")

### Step 8 (to complete) · Load the data into the dataflow.

The Transfer service processes the data asynchronously: the submission returns the id of the request, and its outcome is requested afterwards.

**Hint:** If `dataset` is not None, call `stat_uploader().submit_data(dataset)`: the result has a `request_id`. Then call `stat_uploader().submission_status(request_id, wait=True)`, and print the `outcome` and the `logs` of its result.

In [ ]:
# To be completed

### Step 9 · Read the data back from .Stat.

`stat_data` queries the data of the dataflow on .Stat.

The dataflow and its data can also be seen in the Data Lifecycle Manager of the instance, until the loop is closed in Part 4.

In [ ]:
print("Data Lifecycle Manager:", DLM)
if dataset is not None:
    print(stat_data(flow).tail())

### Step 10 · Create a difference between the FMR and .Stat.

`legacy` is submitted to .Stat only.

In [ ]:
retired = [s for s in legacy if s.short_urn not in before]
if retired:
    result = stat_uploader().submit_structure(retired, STAT_FORMAT)
    print(result.success, result.messages)

## Part 3 · Bring .Stat back in sync

### Step 11 · List the structures on both sides.

`list_urns` requests the structures of the agency one type at a time, as stubs, and returns the short URNs of those whose id starts with `PREFIX`.

The FMR is the reference: without valid credentials its list is empty, and every structure on .Stat would then be taken as retired, so the step stops in that case.

The structures that .Stat had before the exercise was run are left out of the comparison.

In [ ]:
fmr_urns = list_urns(fmr_service())
if not fmr_urns:
    raise RuntimeError("The FMR list is empty: check the user and password.")
stat_urns = list_urns(stat_service()) - before
print("FMR:  ", sorted(fmr_urns))
print(".Stat:", sorted(stat_urns))

### Step 12 (to complete) · Find what .Stat has and the FMR does not.

The extras must be inspected before anything is deleted: they are the EXR_OLD dataflow, its DSD and CL_UNIT. The codelists shared with EXR are still in the FMR, so they are not part of the extras.

**Hint:** Subtract `fmr_urns` from `stat_urns` and print the result.

**Result:** `extras`.

In [ ]:
# To be completed

### Step 13 · Try to delete the DSD first.

.Stat refuses (409), because the EXR_OLD dataflow still uses the DSD.

In [ ]:
try:
    stat_uploader().delete_structure(
        [u for u in extras if u.startswith("DataStructure=")]
    )
except Invalid as error:
    print(error)

### Step 14 (to complete) · Delete the extras, dependents first.

`delete_structure` takes a list of short URNs, deletes them in order and stops at the first error. In that case, fix the cause and run it again with the remaining structures.

**Hint:** `DELETION_ORDER` gives the position of each type (constraints, dataflows, DSDs, concept schemes, codelists). Sort `extras` with the key `DELETION_ORDER[urn.split("=")[0]]` and pass the list to `stat_uploader().delete_structure`.

In [ ]:
# To be completed

### Step 15 · Check that .Stat matches the FMR again.

The printed list must be empty: nothing extra is left on .Stat.

In [ ]:
stat_after = list_urns(stat_service()) - before
print(sorted(stat_after - fmr_urns))

## Part 4 · Close the loop

### Step 16 · Delete the data and the constraint that .Stat created for them.

The exercise now removes what it has published, so that .Stat is left as it was found. The order matters: first the data, then their constraint, and then the structures, dependents first. .Stat refuses to delete a dataflow while its constraint exists.

The structures that the exercise has published are those that .Stat has now and did not have before. `delete_data` submits the observations of the dataflow with the Delete action.

After an interrupted run, the structures that it published are part of `before` on the next run, and are therefore not removed. To remove them, set `before = set()` before running the steps of this part: everything with the prefix is then removed from .Stat.

In [ ]:
published = list_urns(stat_service()) - before
if flow.short_urn in published:
    request = stat_uploader().delete_data(build_dataset(graph))
    status = stat_uploader().submission_status(request.request_id, wait=True)
    print("data deleted:", status.outcome)
    constraint = (
        f"ContentConstraint={flow.agency}:CR_A_{flow.id}({flow.version})"
    )
    with suppress(NotFound):
        for result in stat_uploader().delete_structure(constraint):
            print(result.messages)

### Step 17 · Delete the structures that the exercise has published, dependents first.

`cleanup_dsd` removes what the Transfer service keeps for the DSD.

The printed list must be empty: .Stat is left as it was found. The copies in the FMR remain, because pysdmx cannot delete from the FMR.

In [ ]:
ordered = sorted(published, key=lambda urn: DELETION_ORDER[urn.split("=")[0]])
for result in stat_uploader().delete_structure(ordered):
    print(result.messages)
dsd = graph.get_data_structure_definitions()[0]
if dsd.short_urn in published:
    print(stat_uploader().cleanup_dsd(dsd).message)
print(sorted(list_urns(stat_service()) - before))

### Bonus · Create another difference and remove it.

A codelist that only exists on .Stat is found by the comparison, like the structures in `legacy`.

**Hint:** Build a `Codelist` with `agency=AGENCY`, an id that starts with `PREFIX` and an underscore, a name and two `Code` items, each with an id and a name. Submit it to .Stat only, and repeat the comparison and the deletion.

In [ ]:
# To be completed